In [68]:
import torch

# Load the two tensors
features = torch.load("/home/michele/code/features.pth", map_location=torch.device('cpu'))
coors = torch.load("/home/michele/code/coors.pth", map_location=torch.device('cpu'))

# Define the pooling type
pooling = "max"
# pooling = "avg"

In [69]:
print(f"Features:\n{features}\nshape {features.shape}")
print(f"\nCoors:\n{coors}\nshape {coors.shape}")



# Get the unique coors (no repeated elements) and the count of the elements
voxel_coors, inverse_indices, counts = torch.unique(coors, dim=0, return_counts=True, return_inverse=True)
# Count the "strange" coors (they had too many points, and they have -1 in the index)
counter = 0
for i in range(voxel_coors.shape[0]):
    if voxel_coors[i][0]<0 or voxel_coors[i][1]<0 or voxel_coors[i][2]<0:
        counter+=1
    else:
        break



print(f"\n\n\nUnique_coors:\n{voxel_coors}\nshape {voxel_coors.shape}")
print(f"\nCounts:\n{counts}\nshape {counts.shape}\nand total {torch.sum(counts)}")
print(f"\nInverse_indices:\n{inverse_indices}\nshape {inverse_indices.shape}")

Features:
tensor([[ 6.9627, 10.9242, -0.2260,  0.1326],
        [ 6.9552, 10.8621, -0.2255,  0.0954],
        [ 6.9531, 10.8075, -0.2254,  0.0906],
        ...,
        [-2.7026, -0.3259,  0.2296,  0.0251],
        [-2.7030, -0.3213,  0.2299,  0.0256],
        [-2.7117, -0.3174,  0.2300,  0.0168]])
shape torch.Size([90469, 4])

Coors:
tensor([[  0, 248, 606],
        [  0, 247, 605],
        [  0, 247, 605],
        ...,
        [  0, 177, 545],
        [  0, 177, 545],
        [  0, 178, 545]], dtype=torch.int32)
shape torch.Size([90469, 3])



Unique_coors:
tensor([[ -1,  -1,  -1],
        [ -1,  -1,   0],
        [ -1,   0,   0],
        ...,
        [  0, 359, 824],
        [  0, 359, 825],
        [  0, 359, 827]], dtype=torch.int32)
shape torch.Size([15409, 3])

Counts:
tensor([  494, 13167,  2321,  ...,     1,     1,     1])
shape torch.Size([15409])
and total 90469

Inverse_indices:
tensor([7551, 7501, 7501,  ..., 4841, 4841, 4871])
shape torch.Size([90469])


In [70]:
# Start max-pooling
if pooling == "max":
    # Create a tensor with all values to -inf (so that max-pooling works properly)
    voxel_feats = torch.full((voxel_coors.shape[0], features.shape[1]),   float('-inf'))
    # Cycle trough the inverse_indices (or through the points/features)
    for i in range(inverse_indices.shape[0]):
        
        # If the inverse_index sends back to an "invalid" coors, then just do zero-padding
        if inverse_indices[i]<counter:
            # This if is just for more efficiency (if the value has already been updated, skip)
            if voxel_feats[inverse_indices[i]][0] == float('-inf'):
                # Do the zero-padding for all the depth of the point/feature
                for j in range(voxel_feats.shape[1]):
                    voxel_feats[inverse_indices[i]][j] = 0.0
        
        # Otherwise, do the max-check (and substitute if needed)
        else:
            for j in range(voxel_feats.shape[1]):
                if features[i][j] > voxel_feats[inverse_indices[i]][j]:
                    voxel_feats[inverse_indices[i]][j] = features[i][j]


# Start average-pooling
elif pooling == "avg":
    # Create a tensor with all values to 0 (so that it does not influence the average-pooling)
    voxel_feats = torch.zeros(voxel_coors.shape[0], features.shape[1])
    # Cycle trough the inverse_indices (or through the points/features)
    for i in range(inverse_indices.shape[0]):
        
        # If the inverse_index sends back to an "invalid" coors, then just do zero-padding
        if inverse_indices[i]<counter:
            # Here, no need to do zero-padding since everything is already at zero since the beginning
            continue
        
        # Otherwise, add the average
        else:
            for j in range(voxel_feats.shape[1]):
                voxel_feats[inverse_indices[i]][j] += (features[i][j]/counts[inverse_indices[i]])


print(voxel_feats)
print("\n\n\n")
for i in range(30):
    print(voxel_feats[i])
print("\n\n\n")
for i in range(30):
    print(voxel_feats[-i])

tensor([[ 0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000],
        ...,
        [41.9188, 28.6935,  2.3460,  0.0758],
        [42.1482, 28.7231,  2.3577,  0.0647],
        [42.4441, 28.7930,  2.3727,  0.0714]])




tensor([0., 0., 0., 0.])
tensor([0., 0., 0., 0.])
tensor([0., 0., 0., 0.])
tensor([ 6.1712e+01, -1.9072e+01,  4.9668e+00,  2.4323e-02])
tensor([ 6.0168e+01, -1.9011e+01,  4.9305e+00,  5.7983e-02])
tensor([ 62.0114, -19.0199,   4.9906,   0.0735])
tensor([ 5.8370e+01, -1.8436e+01,  4.7109e+00,  4.3671e-02])
tensor([ 6.5978e+01, -1.8487e+01,  4.8485e+00,  6.4056e-02])
tensor([ 6.6464e+01, -1.8476e+01,  4.8799e+00,  2.1698e-02])
tensor([ 6.8642e+01, -1.8459e+01,  4.9258e+00,  2.3041e-02])
tensor([ 6.9292e+01, -1.8476e+01,  4.9676e+00,  3.5126e-02])
tensor([ 58.5444, -18.3542,   4.7251,   0.0984])
tensor([ 5.8937e+01, -1.8344e+01,  4.7561e+00,  5.5450e-02])
tensor([ 64.7290, -18.3825,   4.8436,   0.079